# dK-series Tutorial

This notebook demonstrates the basic usage of the `dk_series` Python package.

## What is dK-series?

dK-series is a family of network randomization methods that preserve increasing levels of structural properties,
controlled by the parameter `d`.

| d | Preserved statistics |
|---|---|
| 0 | Edge count only (fully random) |
| 1 | Degree distribution P(k) |
| 1.5 | Degree distribution P(k) + clustering coefficient (approximate) |
| 2 | Degree distribution P(k) + joint degree distribution P(k,l) |
| 2.5 | Degree distribution P(k) + P(k,l) + clustering coefficient (approximate) |

## Table of Contents

1. Loading a network
2. Running randomization
3. Comparing statistics
4. Generating multiple randomized networks
5. Converting to NetworkX / igraph
6. Saving randomized networks
7. Visualization


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dk_series

print('dk_series loaded successfully')


---
## 1. Loading a Network

`read_network()` reads an edge-list file where each line contains two integers `u v` representing an edge.
Lines starting with `#` and blank lines are ignored.

```
# comment line
0 1
0 4
1 2
```

Node IDs do not need to start at 0 or be consecutive.


In [ ]:
edges, node_map = dk_series.read_network('data/soc-karate.txt')

N = len(node_map)
M = len(edges)
print(f'Number of nodes: {N}')
print(f'Number of edges: {M}')
print(f'edges  — type: {type(edges)}, shape: {edges.shape}')
print(f'node_map — type: {type(node_map)}, shape: {node_map.shape}')
print()
print('First 5 edges (internal indices):')
print(edges[:5])
print()
print('node_map (internal index -> original node ID):')
print(node_map[:10], '...')


`edges` uses internal indices 0..N-1. The original node IDs are stored in `node_map`,
where `node_map[i]` gives the original ID for internal node `i`.


---
## 2. Running Randomization

Use `randomize(edges, d, seed)` to generate a randomized network.
Setting `seed` ensures reproducibility.

### The `simple` argument

All d values support a `simple` argument (default `False`):
- `simple=False`: faster, may produce self-loops or multi-edges
- `simple=True`: guaranteed simple graph (no self-loops, no multi-edges)

| d | Algorithm (`simple=False`) | Algorithm (`simple=True`) |
|---|---|---|
| 0 | Random (u, v) pairs | Erdős-Rényi G(N, M) — rejection sampling |
| 1 | Configuration model (stubs) | Exact sampling — Del Genio et al. (2010) |
| 1.5 | Edge rewiring (no duplicate check) | Edge rewiring (skips duplicate-creating swaps) |
| 2 | Stub-list per degree class | Exact JDM sampling — Bassler et al. (2015) |
| 2.5 | d=2 (non-simple) + edge rewiring | d=2 (simple) + edge rewiring (skips duplicates) |

**d=0, simple=True**: Erdős-Rényi G(N, M) — uniformly random simple graph with N nodes and M edges.

**d=1, simple=True**: Uses the exact sampling algorithm of Del Genio et al. (2010) [2].

**d=2, simple=True**: Uses the algorithm of Bassler et al. (2015) [3] to exactly sample
a simple graph whose joint degree matrix (JDM) matches the original.


In [ ]:
# d=0: preserve edge count only (fully random)
rand_d0 = dk_series.randomize(edges, d=0, seed=42)
print(f'd=0: {len(rand_d0)} edges')


In [ ]:
# d=1: exact sampling — preserves degree distribution P(k) exactly
#       output is guaranteed to be a simple graph
rand_d1 = dk_series.randomize(edges, d=1, seed=42)
print(f'd=1: {len(rand_d1)} edges')


In [ ]:
# d=2: exact sampling — preserves joint degree distribution P(k,l) exactly
#       uses the algorithm of Bassler et al. (2015)
#       output is guaranteed to be a simple graph
rand_d2 = dk_series.randomize(edges, d=2, seed=42)
print(f'd=2: {len(rand_d2)} edges')


In [ ]:
# d=2.5: preserve P(k,l) + clustering coefficient (approximate)
rand_d25 = dk_series.randomize(edges, d=2.5, seed=42)
print(f'd=2.5: {len(rand_d25)} edges')


### Verifying simple graph properties

We can confirm that d=1 and d=2 produce simple graphs by checking for self-loops and multi-edges.


In [ ]:
def check_simple_graph(edges, label):
    self_loops = [(u, v) for u, v in edges if u == v]
    edge_set = set()
    multi_edges = []
    for u, v in edges:
        key = (min(u, v), max(u, v))
        if key in edge_set:
            multi_edges.append(key)
        edge_set.add(key)
    status = 'simple graph' if not self_loops and not multi_edges else 'NOT a simple graph'
    print(f'{label}: {status} '
          f'(self-loops={len(self_loops)}, multi-edges={len(multi_edges)})')

# Verify simple=True outputs
rand_d1_true = dk_series.randomize(edges, d=1, seed=42, simple=True)
rand_d2_true = dk_series.randomize(edges, d=2, seed=42, simple=True)

check_simple_graph(rand_d0,       'd=0        ')
check_simple_graph(rand_d1_true,  'd=1 simple=True ')
check_simple_graph(rand_d2_true,  'd=2 simple=True ')
check_simple_graph(rand_d25,      'd=2.5      ')


### Using `simple=False` for faster generation

By default `simple=True` and the output is guaranteed to be a simple graph.
Setting `simple=False` uses a faster configuration-model approach for d=1 and d=2,
but the output may contain self-loops or multi-edges.
This option has no effect for d=0, 1.5, and 2.5.


In [ ]:
# Default (simple=False): faster, may produce self-loops or multi-edges
rand_d1_ns = dk_series.randomize(edges, d=1, seed=42)
rand_d2_ns = dk_series.randomize(edges, d=2, seed=42)

# simple=True: guaranteed simple graph
rand_d1_s = dk_series.randomize(edges, d=1, seed=42, simple=True)
rand_d2_s = dk_series.randomize(edges, d=2, seed=42, simple=True)

print('--- simple=False (default) ---')
check_simple_graph(rand_d1_ns, 'd=1')
check_simple_graph(rand_d2_ns, 'd=2')

print('\n--- simple=True ---')
check_simple_graph(rand_d1_s, 'd=1')
check_simple_graph(rand_d2_s, 'd=2')


---
## 3. Comparing Statistics

`compare()` measures how well each randomization preserves the original network's statistics.
Three metrics are computed:

- **P(k) L1**: L1 distance of degree distributions — should be 0 for d≥1
- **P(k,l) L1**: normalized L1 distance of joint degree distributions — should be 0 for d≥2
- **DDCC L1**: normalized L1 distance of degree-dependent clustering coefficients


In [ ]:
print('=== d=0 ===')
dk_series.compare(edges, rand_d0)

print('\n=== d=1 ===')
dk_series.compare(edges, rand_d1)

print('\n=== d=2 ===')
dk_series.compare(edges, rand_d2)

print('\n=== d=2.5 ===')
dk_series.compare(edges, rand_d25);


The return value is a dictionary, so you can access individual metrics programmatically.


In [ ]:
# Summary table across all d values
results = {
    'd=0':   dk_series.compare(edges, rand_d0,  verbose=False),
    'd=1':   dk_series.compare(edges, rand_d1,  verbose=False),
    'd=2':   dk_series.compare(edges, rand_d2,  verbose=False),
    'd=2.5': dk_series.compare(edges, rand_d25, verbose=False),
}

print(f"{'d':>6}  {'P(k) L1':>10}  {'P(k,l) L1':>12}  {'DDCC L1':>10}")
print('-' * 46)
for label, r in results.items():
    print(f"{label:>6}  {r['degree_dist_l1']:>10.6f}  {r['jdm_l1']:>12.6f}  {r['ddcc_l1']:>10.6f}")


---
## 4. Generating Multiple Randomized Networks

Use the `num` argument to generate several independent randomized networks at once.
`compare_multiple()` reports the mean and standard deviation of each metric across all samples.


In [ ]:
# Generate 10 independent randomized networks with d=2
rand_list = dk_series.randomize(edges, d=2, seed=0, num=10)
print(f'Number of networks: {len(rand_list)}')
print(f'Edge counts: {[len(r) for r in rand_list]}')


In [ ]:
summary, _ = dk_series.compare_multiple(edges, rand_list)


---
## 5. Converting to NetworkX / igraph

The edge arrays returned by `randomize()` can be converted to NetworkX or igraph graph objects
for further analysis.


In [ ]:
import networkx as nx

G_orig = dk_series.to_networkx(edges, node_map)
G_rand = dk_series.to_networkx(rand_d2, node_map)

print('=== Original network ===')
print(f'  Nodes: {G_orig.number_of_nodes()}')
print(f'  Edges: {G_orig.number_of_edges()}')
print(f'  Average clustering coefficient: {nx.average_clustering(G_orig):.4f}')
print(f'  Average shortest path length:   {nx.average_shortest_path_length(G_orig):.4f}')
print()
print('=== d=2 randomized network ===')
print(f'  Nodes: {G_rand.number_of_nodes()}')
print(f'  Edges: {G_rand.number_of_edges()}')
print(f'  Average clustering coefficient: {nx.average_clustering(G_rand):.4f}')
print(f'  Average shortest path length:   {nx.average_shortest_path_length(G_rand):.4f}')


In [ ]:
import igraph as ig

g_orig = dk_series.to_igraph(edges, N=N, node_map=node_map)
g_rand = dk_series.to_igraph(rand_d2, N=N, node_map=node_map)

print('=== Original network ===')
print(f'  Clustering coefficient: {g_orig.transitivity_undirected():.4f}')
print(f'  Average path length:    {g_orig.average_path_length():.4f}')
print()
print('=== d=2 randomized network ===')
print(f'  Clustering coefficient: {g_rand.transitivity_undirected():.4f}')
print(f'  Average path length:    {g_rand.average_path_length():.4f}')


---
## 6. Saving Randomized Networks

`write_network()` writes an edge list to a file using the original node IDs.


In [ ]:
import os

os.makedirs('outputs', exist_ok=True)

# Save a single network
dk_series.write_network('outputs/karate_d2_1.txt', rand_d2, node_map)
print('Saved: outputs/karate_d2_1.txt')

# Save multiple networks
for i, r in enumerate(rand_list, start=1):
    dk_series.write_network(f'outputs/karate_d2_{i}.txt', r, node_map)
print('Saved 10 files to outputs/')


---
## 7. Visualization

Compare the degree distributions and joint degree matrices of the original
and randomized networks across all d values.


In [ ]:
from collections import Counter

def get_degree_dist(edges, N):
    deg = np.zeros(N, dtype=int)
    for u, v in edges:
        deg[u] += 1
        deg[v] += 1
    return Counter(deg.tolist())

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
r_edges = [rand_d0, rand_d1, rand_d2, rand_d25]
titles  = ['d=0', 'd=1', 'd=2', 'd=2.5']

orig_dist = get_degree_dist(edges, N)
ks_orig = sorted(orig_dist)
pk_orig = [orig_dist[k] / N for k in ks_orig]

for ax, r, title in zip(axes.flat, r_edges, titles):
    rand_dist = get_degree_dist(r, N)
    ks_rand = sorted(rand_dist)
    pk_rand = [rand_dist[k] / N for k in ks_rand]
    ax.plot(ks_orig, pk_orig, 'o-', label='Original', color='steelblue')
    ax.plot(ks_rand, pk_rand, 's--', label=f'Randomized ({title})', color='tomato', alpha=0.8)
    ax.set_xlabel('Degree k')
    ax.set_ylabel('P(k)')
    ax.set_title(f'Degree distribution — {title}')
    ax.legend()
    ax.set_yscale('log')
    ax.set_xscale('log')

fig.suptitle('Degree distributions: original vs. randomized', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
def get_jdm_matrix(edges, N):
    deg = np.zeros(N, dtype=int)
    for u, v in edges:
        deg[u] += 1
        deg[v] += 1
    max_k = int(deg.max())
    mat = np.zeros((max_k + 1, max_k + 1), dtype=float)
    for u, v in edges:
        k, l = int(deg[u]), int(deg[v])
        mat[k, l] += 1
        mat[l, k] += 1
    total = mat.sum()
    return mat / total if total > 0 else mat

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
jdm_orig = get_jdm_matrix(edges, N)
jdm_d1   = get_jdm_matrix(rand_d1, N)
jdm_d2   = get_jdm_matrix(rand_d2, N)
max_k = jdm_orig.shape[0]
vmax  = jdm_orig.max()

for ax, mat, title in zip(axes, [jdm_orig, jdm_d1, jdm_d2], ['Original', 'd=1', 'd=2']):
    padded = np.zeros((max_k, max_k))
    s = min(mat.shape[0], max_k)
    padded[:s, :s] = mat[:s, :s]
    im = ax.imshow(padded, origin='lower', vmin=0, vmax=vmax, cmap='hot_r')
    ax.set_title(f'Joint degree matrix — {title}')
    ax.set_xlabel('Degree l')
    ax.set_ylabel('Degree k')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()
print('With d=2, the joint degree matrix is preserved exactly.')


---
## Summary

| Operation | Function |
|---|---|
| Load network | `dk_series.read_network(filepath)` |
| Randomize | `dk_series.randomize(edges, d, seed, num)` |
| Compare statistics | `dk_series.compare(orig, rand)` |
| Compare multiple | `dk_series.compare_multiple(orig, rand_list)` |
| Convert to NetworkX | `dk_series.to_networkx(edges, node_map)` |
| Convert to igraph | `dk_series.to_igraph(edges, N, node_map)` |
| Save to file | `dk_series.write_network(filepath, edges, node_map)` |

For more details, see `README.md` and the docstrings of each function.

## References

[1] Orsini, C., Dankulov, M., Colomer-de-Simón, P. et al. Quantifying randomness in real networks. *Nat. Commun.* 6, 8627 (2015). https://doi.org/10.1038/ncomms9627

[2] Del Genio, C. I., Kim, H., Toroczkai, Z., & Bassler, K. E. Efficient and exact sampling of simple graphs with given arbitrary degree sequence. *PLOS ONE*, 5(4), e10012 (2010). https://doi.org/10.1371/journal.pone.0010012

[3] Bassler, K. E., Del Genio, C. I., Erdős, P. L., Miklós, I., & Toroczkai, Z. Exact sampling of graphs with prescribed degree correlations. *New Journal of Physics*, 17(8), 083052 (2015). https://doi.org/10.1088/1367-2630/17/8/083052
